In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lead, desc
import pyspark.sql.functions as F
from pyspark.sql.functions import broadcast

class Transformer:

  def __init__(self):
    pass

  def extract(self):
    pass

class AirpodsAfterIphoneTransformer(Transformer):

  def transform(self,inputDFs):
     
    '''
    Customer who have bought Airpods just after buying iphone 
    '''
     
    transactionInputDf = inputDFs.get("transcatioInputDF")

    print("transactioninputDF in transform")

    #transactionInputDf.show()

    WindowSpec = Window.partitionBy("customer_id").orderBy('transaction_date')

    transactionInputDf = transactionInputDf.withColumn("Next Product Purchase", lead(col("product_name")).over(WindowSpec))

    # transactionInputDf.orderBy("customer_id","transaction_date").show()

    transactionFilteredDf = transactionInputDf.filter((col("product_name") == "iPhone") & (col("Next Product Purchase") == "AirPods"))

    transactionFilteredDf.orderBy("customer_id","transaction_date","product_name").show()
    
    customerInputDF = inputDFs.get("customerInputDF")

    #Plain join
    #joinDF = transactionFilteredDf.join(customersDF,transactionFilteredDf.customer_id == customersDF.customer_id,"inner")   
     
    joinDf = transactionFilteredDf.join(F.broadcast(customerInputDF),"customer_id")

    joinDf.select("customer_name","product_name","Next Product Purchase","location").show()  
  


class OnlyAirpodsAndIphone(Transformer):

    def transform(self, inputDFs):
        """
        Customer who have bought only iPhone and Airpods nothing else
        """

        transcatioInputDF = inputDFs.get("transcatioInputDF")

        print("transcatioInputDF in transform")

        groupedDF = transcatioInputDF.groupBy("customer_id").agg(
            F.collect_set("product_name").alias("products")
        )

        print("Grouped DF")
        groupedDF.show()

        filteredDF = groupedDF.filter(
            (F.array_contains(col("products"), "iPhone")) &
            (F.array_contains(col("products"), "AirPods")) & 
            (F.size(col("products")) == 2)
        )
        
        print("Only Airpods and iPhone")
        filteredDF.show()

        customerInputDF = inputDFs.get("customerInputDF")

        customerInputDF.show()

        joinDF =  customerInputDF.join(
           broadcast(filteredDF),
            "customer_id"
        )

        print("JOINED DF")
        joinDF.show()

        return joinDF.select(
            "customer_id",
            "customer_name",
            "location"
        )


